# OOP Pipeline: JPG ➜ WebP (đa lõi) → Zip theo dataset → Upload Kaggle → Dọn temp

**Dry run**: chế độ chạy thử — vẫn chuyển JPG→WebP và tạo ZIP để bạn kiểm tra kích thước, **nhưng bỏ qua upload Kaggle** và **không xóa thư mục tạm**, giúp bạn xem kết quả trước khi chạy thật.


## Cài đặt môi trường
Nếu cần:

```bash
!pip install kaggle pillow tqdm
```


In [ ]:
!pip install kaggle pillow tqdm

In [ ]:
import os
os.environ["KAGGLE_USERNAME"] = "doragaming"
os.environ["KAGGLE_KEY"] = "eaa9db41048f4b0cecba02981a2fcb21"


In [ ]:
# Cấu hình mặc định — CHỈNH TẠI ĐÂY
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')               # Thư mục gốc chứa nhiều dataset
TEMP_ROOT = Path('/kaggle/temp/webp_zip_upload') # Thư mục tạm để chứa WebP/ZIP
KAGGLE_OWNER = 'doragaming'            # Username Kaggle của bạn

WEBP_QUALITY = 80    # 0-100 (lossy)
WEBP_METHOD = 6      # 0-6
WEBP_LOSSLESS = False

N_PROCESSES = None   # None = dùng hết lõi CPU; hoặc đặt số cụ thể
DRY_RUN = False      # True = chạy thử (không upload & không xóa temp)


In [ ]:
# Triển khai dạng class (OOP)
import os
import re
import json
import shutil
import subprocess
from dataclasses import dataclass
from typing import Optional, Tuple, List
from pathlib import Path

from PIL import Image
from tqdm import tqdm
from multiprocessing import Pool, cpu_count


def _slugify(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r'[^a-z0-9\-]+', '-', s)
    s = re.sub(r'-+', '-', s).strip('-')
    return s

def _strip_batch_suffix(dataset_name: str) -> str:
    # bỏ đuôi -batch-<số>
    return re.sub(r'-batch-\d+$', '', dataset_name)

def _run_cmd(cmd: List[str]) -> int:
    print('> $', ' '.join(cmd))
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(proc.stdout)
    return proc.returncode

def _zip_dir(src_dir: Path, zip_path: Path) -> Path:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    base = zip_path.with_suffix('')
    if base.exists():
        if base.is_dir():
            shutil.rmtree(base)
        else:
            base.unlink()
    shutil.make_archive(str(base), 'zip', root_dir=src_dir)
    return base.with_suffix('.zip')

def _ensure_kaggle_ready():
    try:
        import kaggle  # noqa: F401
    except Exception as e:
        raise RuntimeError("Chưa cài gói 'kaggle'. Hãy chạy: !pip install kaggle") from e
    has_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    has_file = kaggle_json.exists()
    if not (has_env or has_file):
        raise RuntimeError("Thiếu Kaggle credential. Tạo ~/.kaggle/kaggle.json hoặc set KAGGLE_USERNAME/KAGGLE_KEY.")

def _write_dataset_metadata(pkg_dir: Path, owner: str, slug: str, title: Optional[str] = None):
    pkg_dir.mkdir(parents=True, exist_ok=True)
    meta = {
        "title": title or slug.replace('-', ' ').title(),
        "id": f"{owner}/{slug}",
        "licenses": [{"name": "CC0-1.0"}]
    }
    (pkg_dir / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')

def _kaggle_dataset_exists(owner: str, slug: str) -> bool:
    code = _run_cmd(['kaggle', 'datasets', 'status', f'{owner}/{slug}'])
    return code == 0

def _kaggle_upload_zip(owner: str, slug: str, zip_file: Path, message: str = "Update webp"):
    _ensure_kaggle_ready()
    pkg_dir = zip_file.parent / f'kaggle_pkg_{slug}'
    if pkg_dir.exists():
        shutil.rmtree(pkg_dir)
    pkg_dir.mkdir(parents=True, exist_ok=True)
    _write_dataset_metadata(pkg_dir, owner, slug, title=slug.replace('-', ' ').title())
    shutil.copy2(zip_file, pkg_dir / zip_file.name)

    if _kaggle_dataset_exists(owner, slug):
        print(f"Dataset đã tồn tại: {owner}/{slug} → tạo version mới")
        code = _run_cmd(['kaggle', 'datasets', 'version', '-p', str(pkg_dir), '-m', message, '--dir-mode', 'zip'])
    else:
        print(f"Tạo dataset mới: {owner}/{slug}")
        code = _run_cmd(['kaggle', 'datasets', 'create', '-p', str(pkg_dir), '--dir-mode', 'zip'])

    try:
        shutil.rmtree(pkg_dir)
    except Exception as e:
        print('Không thể xóa pkg_dir:', e)
    if code != 0:
        raise RuntimeError("Upload Kaggle thất bại. Kiểm tra log ở trên.")
    print('Upload Kaggle thành công!')

@dataclass
class WebPConfig:
    quality: int = 80
    method: int = 6
    lossless: bool = False

@dataclass
class PipelineConfig:
    input_root: Path
    temp_root: Path
    kaggle_owner: str
    n_processes: Optional[int] = None
    dry_run: bool = False
    valid_exts: tuple = ('.jpg')
    chunksize: int = 64

class KaggleWebPPipeline:
    def __init__(self, pipe_cfg: PipelineConfig, webp_cfg: WebPConfig):
        self.cfg = pipe_cfg
        self.webp = webp_cfg
        self.processed_root = self.cfg.temp_root / 'processed'
        self.zips_root = self.cfg.temp_root / 'zips'
        self.processed_root.mkdir(parents=True, exist_ok=True)
        self.zips_root.mkdir(parents=True, exist_ok=True)

    # ---- Static/worker for Pool
    @staticmethod
    def _process_one_image(job) -> Tuple[int, int, int, str]:
        src, dst, quality, method, lossless = job
        try:
            from PIL import Image
            dst.parent.mkdir(parents=True, exist_ok=True)
            with Image.open(src) as im:
                if im.mode in ('P', 'LA'):
                    im = im.convert('RGBA')
                elif im.mode not in ('RGB', 'RGBA'):
                    im = im.convert('RGB')
                before = src.stat().st_size if src.exists() else 0
                im.save(dst, format='WEBP', quality=quality, method=method, lossless=lossless)
                after = dst.stat().st_size if dst.exists() else 0
            return (before, after, 1, None)
        except Exception as e:
            return (0, 0, 0, f"{src}: {e}")
    def _copy_metadata_files(self, ds_in, ds_out):
        """Copy metadata.json files from input dataset to output dataset"""
        for vdir in ds_in.iterdir():
            if vdir.is_dir():
                out_vdir = ds_out / vdir.name
                out_vdir.mkdir(parents=True, exist_ok=True)
                
                # Copy metadata.json if it exists
                metadata_file = vdir / "metadata.json"
                if metadata_file.exists():
                    target_metadata = out_vdir / "metadata.json"
                    shutil.copy2(metadata_file, target_metadata)
                    # print(f"Copied metadata: {metadata_file} -> {target_metadata}")
        
        
    
    def _gather_jobs_for_dataset(self, ds_path: Path, ds_out: Path) -> List[tuple]:
        jobs = []
        for vdir in sorted([p for p in ds_path.iterdir() if p.is_dir()]):
            out_vdir = ds_out / vdir.name
            for p in sorted([x for x in vdir.iterdir() if x.is_file() and x.suffix.lower() in self.cfg.valid_exts]):
                dst = out_vdir / (p.stem + '.webp')
                jobs.append((p, dst, self.webp.quality, self.webp.method, self.webp.lossless))
        return jobs

    def _compress_dataset(self, ds_path: Path) -> tuple:
        ds_name = ds_path.name
        ds_out = self.processed_root / ds_name
        if ds_out.exists():
            shutil.rmtree(ds_out)
        ds_out.mkdir(parents=True, exist_ok=True)
        
        self._copy_metadata_files(ds_path, ds_out)

        jobs = self._gather_jobs_for_dataset(ds_path, ds_out)
        print(f"Tổng số ảnh cần nén ({ds_name}): {len(jobs)}")
        total_in = 0
        total_out = 0
        ok = 0
        err = 0

        if jobs:
            nproc = self.cfg.n_processes or cpu_count()
            print(f"Dùng {nproc} process để nén...")
            with Pool(processes=nproc) as p:
                for before, after, ok1, err_msg in tqdm(
                    p.imap_unordered(self._process_one_image, jobs, chunksize=self.cfg.chunksize),
                    total=len(jobs), desc=f"{ds_name} → Nén song song"
                ):
                    total_in += before
                    total_out += after
                    ok += ok1
                    if err_msg:
                        err += 1
                        if err <= 20:
                            print('Lỗi:', err_msg)

        saved_pct = (1 - (total_out/total_in) if total_in else 0) * 100
        print(f"Nén xong: ảnh OK={ok}, lỗi={err}, trước={total_in/1e6:.2f}MB → sau={total_out/1e6:.2f}MB, tiết kiệm ~{saved_pct:.1f}%")
        return ds_out, ok, err, total_in, total_out

    def _zip_dataset(self, ds_name: str, ds_out: Path) -> Path:
        zip_path = self.zips_root / f"{ds_name}.zip"
        return _zip_dir(ds_out, zip_path)

    def _upload_dataset(self, ds_name: str, zip_path: Path):
        slug = _slugify(_strip_batch_suffix(ds_name))
        _kaggle_upload_zip(owner=self.cfg.kaggle_owner, slug=slug, zip_file=zip_path,
                           message="webp conversion + zipped (OOP parallel)")

    def _cleanup_dataset_temp(self, ds_out: Path, zip_path: Path):
        try:
            if ds_out.exists():
                shutil.rmtree(ds_out)
            if zip_path.exists():
                zip_path.unlink()
        except Exception as e:
            print('Không thể dọn dẹp temp:', e)

    def run(self):
        if not self.cfg.input_root.exists():
            raise FileNotFoundError(f"Không thấy input_root: {self.cfg.input_root}")

        datasets = sorted([p for p in self.cfg.input_root.iterdir() if p.is_dir()])
        if not datasets:
            print('Không có dataset con nào trong', self.cfg.input_root)
            return

        for ds in datasets:
            print('\n==============================')
            print('Dataset:', ds.name)
            ds_out, ok, err, total_in, total_out = self._compress_dataset(ds)
            zip_path = self._zip_dataset(ds.name, ds_out)
            print('Đã tạo ZIP:', zip_path)

            if self.cfg.dry_run:
                print('[DRY RUN] Bỏ qua upload Kaggle & KHÔNG xóa temp (để kiểm tra)')
            else:
                self._upload_dataset(ds.name, zip_path)
                self._cleanup_dataset_temp(ds_out, zip_path)

        print('\nHoàn tất tất cả dataset!')


## Chạy pipeline
Chỉnh cấu hình ở ô trên, sau đó chạy:


In [ ]:
pipe_cfg = PipelineConfig(
    input_root=INPUT_ROOT,
    temp_root=TEMP_ROOT,
    kaggle_owner=KAGGLE_OWNER,
    n_processes=N_PROCESSES,
    dry_run=DRY_RUN,
)
webp_cfg = WebPConfig(
    quality=WEBP_QUALITY,
    method=WEBP_METHOD,
    lossless=WEBP_LOSSLESS,
)

pipeline = KaggleWebPPipeline(pipe_cfg=pipe_cfg, webp_cfg=webp_cfg)
pipeline.run()
